In [1]:
import cdsapi
from tqdm import tqdm
import time
import os
import xarray as xr

In [2]:
def get_request(year,season):
    """
    Get dataset and request form for ERA5 european region

    Parameters
    ----------
    year : int
        The year
    season : str
        Spring or fall.
    """

    if season == "spring":
        months = [
            "01", "02", "03",
            "04", "05", "06"
        ]
    elif season == "autumn":
        months = [
            "07", "08", "09",
            "10", "11", "12"
        ]
    else:
        raise ValueError()

    if not isinstance(year,str): raise ValueError()

    dataset = "reanalysis-era5-single-levels"
    request = {
        "product_type": ["reanalysis"],
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "2m_dewpoint_temperature",
            "2m_temperature",
            # "mean_wave_direction",
            # "mean_wave_period",
            # "significant_height_of_combined_wind_waves_and_swell",
            # "peak_wave_period"
        ],
        "year": [year],
        "month": months,
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30",
            "31"
        ],
        "time": [
            "00:00", "03:00", "06:00",
            "09:00", "12:00", "15:00",
            "18:00", "21:00"
        ],
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": [70, -10, 30, 35]
    }
    return dataset, request

In [ ]:
start_year = 1940
stop_year = 2026
download_folder = "download/"
pbar = tqdm(total=(stop_year-start_year)*2)
if not os.path.exists(download_folder): os.mkdir(download_folder)
for year in range(start_year,stop_year):
    year = f"{year}"
    for season in ["spring","autumn"]:
        if not os.path.exists(download_folder+f"ERA5_{year}_{season}.nc"):
            dataset,request = get_request(year,season)
            pbar.set_description(f"{year}-{season}")
            client = cdsapi.Client()
            client.retrieve(dataset, request).download()
            time.sleep(1)
            new_file = [f for f in os.listdir() if f.endswith(".nc")]
            if len(new_file) == 0:
                raise ValueError("No file found!")
            if len(new_file) > 1:
                raise ValueError("Too many files!")
            os.rename(new_file[0],download_folder+f"ERA5_{year}_{season}.nc")
            time.sleep(1)
        pbar.update()

2025-autumn: 100%|██████████| 172/172 [2:14:56<00:00, 47.07s/it] 
